# Macroscopic observables

We compare the fixed points of the resistor perceptron at $\kappa=1.26,2.56$
with unconstrained linear regression. The calculations use $N=80$, $r=\sigma=1$,
and 20 realizations at each of 40 values of $\alpha=N/P$. Generate the three
endpoint ensembles in `REPRODUCING.txt`, then run the cells below in order.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from resistor_theory import predict

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/resistor_theory").is_dir())
results = root.parent / "results/public-release/finite-n"
datasets = []
paired = None
for kappa, name, table in (
    (1.26, "N80_sweep_alpha40_kappa1.26", "endpoints.csv"),
    (2.56, "N80_sweep_alpha40_kappa2.56", "endpoints.csv"),
    (np.inf, "N80_sweep_alpha40_linear_gd", "linear_endpoints.csv"),
):
    path = results / name / table
    if not path.is_file():
        raise FileNotFoundError("Generate the N=80 endpoints using REPRODUCING.txt first.")
    with path.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    keys = {(int(row["P"]), int(row["seed"])): row["problem_fingerprint"] for row in rows}
    if len(rows) != 800 or len(keys) != 800 or {int(row["N"]) for row in rows} != {80}:
        raise ValueError(f"Expected 800 distinct N=80 endpoints in {path}.")
    if paired is not None and keys != paired:
        raise ValueError("The three ensembles must contain the same datasets.")
    paired = keys
    datasets.append((kappa, rows))

The rows show training loss, risk, self-overlap $q$, and teacher overlap $M$.
Points are individual realizations, colored by the solved regime. Black curves
are the thermodynamic predictions. Since $r=1$, the raw overlaps equal $q/r^2$
and $M/r^2$.

In [ ]:
colors = {"ols": "#0072B2", "lasso": "#D55E00", "bregman": "#009E73",
          "critical": "#7B3294", "minimum_norm_interpolator": "#009E73"}
fig, axes = plt.subplots(4, 3, figsize=(9, 9), sharex=True)
for column, (kappa, rows) in enumerate(datasets):
    for axis, metric, label in zip(axes[:, column], ("loss", "risk", "q", "M"),
                                   (r"$L_{\mathcal{D}}$", r"$\mathcal{R}_N$", r"$q$", r"$M$")):
        for regime in sorted({row["regime"] for row in rows}):
            selected = [row for row in rows if row["regime"] == regime]
            axis.scatter([float(row["alpha"]) for row in selected],
                         [float(row[metric]) for row in selected], s=8, alpha=0.3,
                         color=colors.get(regime, "grey"), label=regime)
        axis.set(xscale="log", ylabel=label)
    for alpha in (np.geomspace(0.1, 0.999, 100), np.geomspace(1.001, 10, 100)):
        theory = [predict(kappa, a, 1.0, 1.0) for a in alpha]
        for axis, metric in zip(axes[:, column], ("loss", "risk", "q", "M")):
            axis.plot(alpha, [value[metric] for value in theory], "k-", lw=1)
    axes[1, column].set_yscale("log")
    axes[2, column].set_yscale("log")
    capacity = r"\infty" if np.isinf(kappa) else str(kappa)
    axes[0, column].set_title(fr"$\kappa={capacity}$")
    axes[-1, column].set_xlabel(r"$\alpha=N/P$")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()